# Notebook 01: Migración del Proyecto 1 a MongoDB

Pasa todo el corpus del Proyecto 1 (6.283 reseñas) a MongoDB, **un documento por reseña**, con el esquema de `docs/esquema_mongodb.md`:

- **Texto y metadatos** de `resenas_limpias.csv`.
- **POS Tags** (spaCy y NLTK) de `pos_analysis.csv.gz`, agrupados por reseña.
- **Métricas morfosintácticas** calculadas por reseña.

La cadena de conexión se lee de la variable de entorno `MONGODB_URI` (archivo `.env`, que debe estar en `.gitignore`). Los embeddings no se calculan aquí: van después en la colección `embeddings`.

## 1. Configuración

In [1]:
import os
import sys

sys.path.append("..")  # por si se usa la estructura del repo (con src/)

try:
    from src.db_manager import (get_db, crear_indices, guardar_resenas, conteo_por,
                                genero_por_origen, calificacion_media_por_genero)
    from src.migracion import leer_resenas, leer_tokens, construir_documentos
except ImportError:  # archivos en la misma carpeta que el notebook
    from db_manager import (get_db, crear_indices, guardar_resenas, conteo_por,
                            genero_por_origen, calificacion_media_por_genero)
    from migracion import leer_resenas, leer_tokens, construir_documentos


def buscar(*nombres):
    """Busca el archivo en esta carpeta, en data/raw y en ../data/raw."""
    for carpeta in (".", "data/raw", "../data/raw"):
        for nombre in nombres:
            ruta = os.path.join(carpeta, nombre)
            if os.path.exists(ruta):
                return ruta
    raise FileNotFoundError(f"No encuentro {nombres[0]}. Ponlo en la misma carpeta del notebook.")


RUTA_RESENAS = buscar("resenas_limpias.csv", "resenas_limpias_2.csv")
RUTA_POS = buscar("pos_analysis.csv.gz", "pos_analysis.csv")
print(RUTA_RESENAS, RUTA_POS)

.\resenas_limpias.csv .\pos_analysis.csv.gz


## 2. Carga de los datos del Proyecto 1

Se esperan **6.283 reseñas** y **1.361.253 tokens**.

In [7]:
resenas = leer_resenas("C:/proyecto2textos/resenas_limpias.csv")
tokens = leer_tokens("C:/proyecto2textos/pos_analysis.csv.gz")

print(resenas.shape, tokens.shape)
assert resenas["id"].is_unique
print("Reseñas sin tokens:", (~resenas["id"].isin(tokens["id"])).sum())

(6283, 11) (1361253, 4)
Reseñas sin tokens: 0


## 3. Construcción de los documentos

Cada reseña se convierte en un documento. Decisiones:

- `genero_principal`: el género del Proyecto 1 (el primero de la lista), traducido al español.
- `origen_texto = "traducido"` y `fuente = "imdb_kaggle"` para todo el corpus del Proyecto 1.
- `id_resena = p1_<id>`: identificador propio que evita duplicados si se vuelve a correr la migración.
- Los campos que el CSV no trae (`titulo_obra`, `anio_estreno`, `url_fuente`) no se guardan: el esquema flexible de MongoDB lo permite.

In [3]:
docs = construir_documentos(resenas, tokens)

ejemplo = {k: v for k, v in docs[0].items() if k not in ("texto_es", "texto_en", "pos_tags")}
print(len(docs), "documentos")
print(ejemplo)
print("tokens en pos_tags:", len(docs[0]["pos_tags"]["spacy"]))

6283 documentos
{'id_resena': 'p1_1', 'origen_texto': 'traducido', 'tipo_obra': 'pelicula', 'generos': ['Animación'], 'genero_principal': 'Animación', 'calificacion': 10, 'polaridad': 'positiva', 'periodo': 'antes_2010', 'fuente': 'imdb_kaggle', 'fecha_resena': datetime.datetime(2002, 5, 5, 0, 0), 'fecha_recopilacion': datetime.datetime(2026, 11, 12, 23, 50, 54, 23232, tzinfo=datetime.timezone.utc), 'movie_id': 'tt0092067', 'anio_review': 2002, 'is_spoiler': False, 'metricas': {'num_palabras': 109, 'densidad_lexica': 0.4495, 'ratio_verbos_sustantivos': 0.75, 'indice_adjetivacion': 5.5046, 'indice_negacion': 0.9174}}
tokens en pos_tags: 125


### (Opcional) Completar título, año de estreno y lista de géneros

Si se quiere completar esos campos, se arma un diccionario `detalles` desde `IMDB_movie_details.json` del dataset de Kaggle (se descarga con `kagglehub`) y se pasa a `construir_documentos(resenas, tokens, detalles)`. Hay que revisar los nombres de columnas del JSON y traducir los géneros al español.

## 4. Carga en MongoDB

In [8]:
SOLO_PRUEBA = False  # True: sube solo 50 reseñas para comprobar. Cámbialo a False para subir todas.

db = get_db()
crear_indices(db)

n = guardar_resenas(db, docs[:50] if SOLO_PRUEBA else docs)
print("Documentos escritos:", n)

Documentos escritos: 6233


## 5. Verificación con consultas

Con `SOLO_PRUEBA = True` deben aparecer 50 documentos. Con `SOLO_PRUEBA = False`, **6.283**, repartidos en 5 géneros y 3 polaridades.

In [9]:
col = db["resenas"]
print("Total:", col.count_documents({}))
print("Por género:", conteo_por(db, "genero_principal"))
print("Por polaridad:", conteo_por(db, "polaridad"))
print("Género x origen:", genero_por_origen(db))
print("Calificación media:", calificacion_media_por_genero(db))

Total: 6283
Por género: [{'_id': 'Drama', 'n': 1261}, {'_id': 'Animación', 'n': 1258}, {'_id': 'Comedia', 'n': 1257}, {'_id': 'Acción', 'n': 1256}, {'_id': 'Terror', 'n': 1251}]
Por polaridad: [{'_id': 'positiva', 'n': 4175}, {'_id': 'negativa', 'n': 1203}, {'_id': 'intermedia', 'n': 905}]
Género x origen: [{'_id': {'genero': 'Acción', 'origen': 'traducido'}, 'n': 1256}, {'_id': {'genero': 'Animación', 'origen': 'traducido'}, 'n': 1258}, {'_id': {'genero': 'Comedia', 'origen': 'traducido'}, 'n': 1257}, {'_id': {'genero': 'Drama', 'origen': 'traducido'}, 'n': 1261}, {'_id': {'genero': 'Terror', 'origen': 'traducido'}, 'n': 1251}]
Calificación media: [{'_id': 'Animación', 'calificacion_media': 7.647058823529412, 'n': 1258}, {'_id': 'Drama', 'calificacion_media': 7.234734337827121, 'n': 1261}, {'_id': 'Comedia', 'calificacion_media': 7.054892601431981, 'n': 1257}, {'_id': 'Terror', 'calificacion_media': 6.677857713828937, 'n': 1251}, {'_id': 'Acción', 'calificacion_media': 6.5955414012738

In [10]:
# Consulta con filtro: reseñas negativas de Terror con spoiler
cursor = col.find(
    {"genero_principal": "Terror", "polaridad": "negativa", "is_spoiler": True},
    {"_id": 0, "id_resena": 1, "calificacion": 1, "metricas.indice_negacion": 1},
).limit(3)
list(cursor)

[{'id_resena': 'p1_156',
  'calificacion': 1,
  'metricas': {'indice_negacion': 6.0606}},
 {'id_resena': 'p1_226',
  'calificacion': 4,
  'metricas': {'indice_negacion': 1.6484}},
 {'id_resena': 'p1_342',
  'calificacion': 3,
  'metricas': {'indice_negacion': 0.9009}}]